# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [2]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (16992, 23)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [3]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5162,7921009.69,1541.05,1207.64,10895.0
Audio,3949,5495070.86,1395.40,1108.27,8314.0
Accessoire,3870,4430807.59,1148.18,884.01,8178.0
Mobilier,3936,4196068.73,1070.15,779.17,8339.0


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [4]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie  montant  ca_categorie  part_dans_categorie
0  Informatique   481.98    7921009.69               0.0061
1      Mobilier   140.96    4196068.73               0.0034
2         Audio  2073.36    5495070.86               0.0377
3  Informatique  2281.00    7921009.69               0.0288
4      Mobilier  2116.10    4196068.73               0.0504

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [5]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,5738,P0027,0.165
telephone,2865,P0015,0.166
web,8389,P0038,0.170


### Partie autonome

In [6]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.

tableau_croise = sales.pivot_table(
    index='categorie',
    columns='canal',
    values='montant',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
    fill_value=0
)

display(tableau_croise)

canal,boutique,telephone,web,Total
categorie,,,,
Accessoire,1417866.65,742808.74,2270132.20,4430807.59
Audio,1760553.16,918100.15,2816417.55,5495070.86
Informatique,2771226.41,1351027.63,3798755.65,7921009.69
Mobilier,1496314.20,692035.68,2007718.85,4196068.73
Total,7445960.42,3703972.20,10893024.25,22042956.87


In [7]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.

stats_clients = sales.groupby('id_client').agg(
    nb_commandes=('id_commande', 'nunique'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique')
)

top_10_clients = stats_clients.sort_values(
    'ca_total',
    ascending=False
).head(10)

display(top_10_clients)

,nb_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,nb_categories
id_client,,,,,,
C00168,12,24838.61,2069.884167,2023-02-01 00:00:00,2024-10-17 00:00:00,3
C00872,10,22875.88,2287.588000,2023-01-21 00:00:00,2024-09-12 14:32:00,4
C00833,12,22291.37,1857.614167,2023-01-30 00:00:00,2024-09-22 00:00:00,4
C01339,10,21774.69,2177.469000,2023-05-20 00:00:00,2024-09-20 00:00:00,4
C02600,11,21645.81,1967.800909,2023-03-17 09:36:00,2024-12-06 00:00:00,4
C00741,15,21184.05,1412.270000,2023-01-13 09:59:00,2024-11-14 00:00:00,4
C02145,13,21038.35,1618.334615,2023-01-12 00:00:00,2024-07-16 00:00:00,4
C01503,12,20761.74,1730.145000,2023-05-30 00:00:00,2024-12-17 00:00:00,4
C01999,11,20579.69,1870.880909,2023-04-04 10:58:00,2024-11-26 00:00:00,4


In [8]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?

sales['mediane_categorie'] = sales.groupby(
    'categorie'
)['montant'].transform('median')

sales['commande_elevee'] = (
    sales['montant'] > 1.5 * sales['mediane_categorie']
)

part_commandes = sales['commande_elevee'].mean() * 100

print(f"Part des commandes élevées : {part_commandes:.2f} %")

display(
    sales[['categorie', 'montant',
           'mediane_categorie', 'commande_elevee']].head(10)
)

Part des commandes élevées : 33.49 %


,categorie,montant,mediane_categorie,commande_elevee
0,Informatique,481.98,1207.64,False
1,Mobilier,140.96,779.17,False
2,Audio,2073.36,1108.27,True
3,Informatique,2281.00,1207.64,True
4,Mobilier,2116.10,779.17,True
5,Audio,1419.04,1108.27,False
6,Mobilier,470.65,779.17,False
7,Informatique,1224.22,1207.64,False
8,Accessoire,2570.70,884.01,True
9,Audio,608.78,1108.27,False


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [9]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 16992
Après jointure produits : 16992
Lignes sans produit correspondant : 75 (0.44%)
Identifiants fautifs : <ArrowStringArray>
['P9999']
Length: 1, dtype: str


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [10]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
16992 lignes -> 17287 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [11]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_client
   C00001
   C00002
   C00003
   C00004
   C00005 ...


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [12]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).

print("Lignes initiales :", len(sales))

df = sales.merge(
    products,
    on='id_produit',
    how='left',
    validate='many_to_one',
    suffixes=('', '_prod')
)
print("Après produits :", len(df))

orphelins = df['libelle'].isna().sum()
print("Produits non trouvés :", orphelins)

df = df.merge(
    customers,
    on='id_client',
    how='left',
    validate='many_to_one',
    suffixes=('', '_client')
)
print("Après clients :", len(df))

df = df.merge(
    stores,
    on='id_magasin',
    how='left',
    validate='many_to_one',
    suffixes=('', '_magasin')
)
print("Après magasins :", len(df))

print("Nombre final de lignes :", len(df))

display(df.head())

Lignes initiales : 16992
Après produits : 16992
Produits non trouvés : 75
Après clients : 16992
Après magasins : 16992
Nombre final de lignes : 16992


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison,quantite_suspecte,montant,libelle,categorie,sous_categorie,cout_achat,marge_unitaire,taux_marge,montant_net,panier_moyen_client,est_non_aboutie,ecart_prix_categorie,ca_categorie,part_dans_categorie,mediane_categorie,commande_elevee,libelle_prod,categorie_prod,sous_categorie_prod,cout_achat_prod,date_inscription,segment,ville,age,nom_magasin,ville_magasin,region,surface_m2
0,CMD0003112,2024-01-18 19:53:00,C02300,P0009,M05,3.0,160.66,5.0,telephone,retourne,lille,False,481.98,Clavier modele 3,Informatique,Clavier,86.41,74.25,0.4622,457.881,1486.903333,1,-0.8138,7921009.69,0.0061,1207.64,False,Clavier modele 3,Informatique,Clavier,86.41,2021-04-18,Particulier,PARIS,39.0,Boutique Bordeaux,Bordeaux,Nouvelle-Aquitaine,769
1,CMD0000107,2024-01-08 00:00:00,C02791,P0028,M03,2.0,70.48,0.0,boutique,livre,toulouse,False,140.96,Lampe modele 1,Mobilier,Lampe,34.81,35.67,0.5061,140.960,930.415714,0,-0.8445,4196068.73,0.0034,779.17,False,Lampe modele 1,Mobilier,Lampe,34.81,2024-02-19,Particulier,PARIS,36.0,Boutique Marseille,Marseille,PACA,460
2,CMD0010050,2023-04-15 00:00:00,990031,P0017,M04,3.0,691.12,0.0,boutique,livre,marseille,False,2073.36,Enceinte modele 2,Audio,Enceinte,347.41,343.71,0.4973,2073.360,1282.200000,0,-0.0518,5495070.86,0.0377,1108.27,True,Enceinte modele 2,Audio,Enceinte,347.41,2019-02-24,Professionnel,Bordeaux,NaN,Boutique Lille,Lille,Hauts-de-France,245
3,CMD0001979,2024-11-06 00:00:00,C01883,P0010,M04,4.0,570.25,0.0,web,retourne,marseille,False,2281.00,Souris modele 1,Informatique,Souris,283.64,286.61,0.5026,2281.000,1256.012500,1,-0.3391,7921009.69,0.0288,1207.64,True,Souris modele 1,Informatique,Souris,283.64,2019-06-01,Particulier,Nantes,77.0,Boutique Lille,Lille,Hauts-de-France,245
4,CMD0010513,2024-02-11 00:00:00,C00571,P0024,M02,2.0,1058.05,10.0,web,livre,marseille,False,2116.10,Bureau modele 3,Mobilier,Bureau,554.59,503.46,0.4758,1904.490,1463.313333,0,1.3345,4196068.73,0.0504,779.17,True,Bureau modele 3,Mobilier,Bureau,554.59,2021-01-02,Particulier,Strasbourg,44.0,Boutique Lyon,Lyon,Auvergne-Rhone-Alpes,853


In [13]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.

def safe_merge(left, right, **kwargs):

    n_before = len(left)

    result = left.merge(
        right,
        indicator=True,
        **kwargs
    )

    n_after = len(result)

    if kwargs.get('how', 'inner') == 'left':
        if n_before != n_after:
            raise ValueError(
                f"Nombre de lignes modifié : "
                f"{n_before} -> {n_after}"
            )

    taux = (
        (result['_merge'] == 'left_only').mean() * 100
    )

    print(f"Taux de clés non appariées : {taux:.2f}%")

    result = result.drop(columns='_merge')

    return result


test = safe_merge(
    sales,
    products,
    on='id_produit',
    how='left',
    validate='many_to_one'
)

display(test.head())

Taux de clés non appariées : 0.44%


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison,quantite_suspecte,montant,libelle_x,categorie_x,sous_categorie_x,cout_achat_x,marge_unitaire,taux_marge,montant_net,panier_moyen_client,est_non_aboutie,ecart_prix_categorie,ca_categorie,part_dans_categorie,mediane_categorie,commande_elevee,libelle_y,categorie_y,sous_categorie_y,cout_achat_y
0,CMD0003112,2024-01-18 19:53:00,C02300,P0009,M05,3.0,160.66,5.0,telephone,retourne,lille,False,481.98,Clavier modele 3,Informatique,Clavier,86.41,74.25,0.4622,457.881,1486.903333,1,-0.8138,7921009.69,0.0061,1207.64,False,Clavier modele 3,Informatique,Clavier,86.41
1,CMD0000107,2024-01-08 00:00:00,C02791,P0028,M03,2.0,70.48,0.0,boutique,livre,toulouse,False,140.96,Lampe modele 1,Mobilier,Lampe,34.81,35.67,0.5061,140.960,930.415714,0,-0.8445,4196068.73,0.0034,779.17,False,Lampe modele 1,Mobilier,Lampe,34.81
2,CMD0010050,2023-04-15 00:00:00,990031,P0017,M04,3.0,691.12,0.0,boutique,livre,marseille,False,2073.36,Enceinte modele 2,Audio,Enceinte,347.41,343.71,0.4973,2073.360,1282.200000,0,-0.0518,5495070.86,0.0377,1108.27,True,Enceinte modele 2,Audio,Enceinte,347.41
3,CMD0001979,2024-11-06 00:00:00,C01883,P0010,M04,4.0,570.25,0.0,web,retourne,marseille,False,2281.00,Souris modele 1,Informatique,Souris,283.64,286.61,0.5026,2281.000,1256.012500,1,-0.3391,7921009.69,0.0288,1207.64,True,Souris modele 1,Informatique,Souris,283.64
4,CMD0010513,2024-02-11 00:00:00,C00571,P0024,M02,2.0,1058.05,10.0,web,livre,marseille,False,2116.10,Bureau modele 3,Mobilier,Bureau,554.59,503.46,0.4758,1904.490,1463.313333,0,1.3345,4196068.73,0.0504,779.17,True,Bureau modele 3,Mobilier,Bureau,554.59


---
## Atelier 6.3 — Formats long et large (35 min)

In [14]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{month:02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,408749.0,309885.0,363209.0,369125.0,366474.0,364975.0,363530.0,244600.0,353740.0,356774.0,454128.0,475617.0
Audio,450253.0,415390.0,428261.0,454111.0,476464.0,452042.0,493188.0,333421.0,453913.0,437570.0,555143.0,545315.0
Informatique,616720.0,635567.0,674976.0,694012.0,638732.0,684281.0,694124.0,449859.0,629719.0,624460.0,760190.0,818371.0
Mobilier,352811.0,333252.0,328214.0,318734.0,359047.0,361624.0,332759.0,222582.0,355577.0,343043.0,434135.0,454291.0


In [15]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,408749.0
1,Audio,mois_01,450253.0
2,Informatique,mois_01,616720.0
3,Mobilier,mois_01,352811.0
4,Accessoire,mois_02,309885.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [16]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.

wide_reconstitue = long.pivot(
    index='categorie',
    columns='mois',
    values='ca'
)

wide_reconstitue = wide_reconstitue[wide.columns]
wide_reconstitue.columns.name = None

print("Tableaux identiques :", wide_reconstitue.equals(wide))

display(wide_reconstitue)

Tableaux identiques : True


,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,408749.0,309885.0,363209.0,369125.0,366474.0,364975.0,363530.0,244600.0,353740.0,356774.0,454128.0,475617.0
Audio,450253.0,415390.0,428261.0,454111.0,476464.0,452042.0,493188.0,333421.0,453913.0,437570.0,555143.0,545315.0
Informatique,616720.0,635567.0,674976.0,694012.0,638732.0,684281.0,694124.0,449859.0,629719.0,624460.0,760190.0,818371.0
Mobilier,352811.0,333252.0,328214.0,318734.0,359047.0,361624.0,332759.0,222582.0,355577.0,343043.0,434135.0,454291.0


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [18]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [20]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.

from pathlib import Path
import pandas as pd

output_dir = Path("data/processed/star")
output_dir.mkdir(parents=True, exist_ok=True)

fact_sales = sales.copy()

fact_sales['cle_date'] = (
    pd.to_datetime(fact_sales['date_commande'])
    .dt.strftime('%Y%m%d')
    .astype(int)
)

colonnes = [
    'cle_date', 'id_client', 'id_produit', 'id_magasin',
    'quantite', 'prix_unitaire', 'remise_pct',
    'montant', 'montant_net', 'statut'
]

fact_sales = fact_sales[colonnes]

dim_customer = customers.drop_duplicates(subset='id_client').copy()
dim_product = products.drop_duplicates(subset='id_produit').copy()
dim_store = stores.drop_duplicates(subset='id_magasin').copy()

dimensions = {
    'cle_date': dim_date,
    'id_client': dim_customer,
    'id_produit': dim_product,
    'id_magasin': dim_store
}

for cle, dimension in dimensions.items():
    manquantes = fact_sales.loc[
        ~fact_sales[cle].isin(dimension[cle]),
        cle
    ].drop_duplicates()

    print(f"{cle} : {len(manquantes)} clés distinctes manquantes")

    if len(manquantes) > 0:
        nouvelles_lignes = pd.DataFrame({cle: manquantes})
        dimension = pd.concat(
            [dimension, nouvelles_lignes],
            ignore_index=True
        )
        dimensions[cle] = dimension

dim_date = dimensions['cle_date']
dim_customer = dimensions['id_client']
dim_product = dimensions['id_produit']
dim_store = dimensions['id_magasin']

for cle, dimension in dimensions.items():
    orphelines = ~fact_sales[cle].isin(dimension[cle])
    assert not orphelines.any(), f"{cle} : {orphelines.sum()} clés orphelines"

fact_sales.to_parquet(output_dir / "fact_sales.parquet", index=False)
dim_date.to_parquet(output_dir / "dim_date.parquet", index=False)
dim_customer.to_parquet(output_dir / "dim_customer.parquet", index=False)
dim_product.to_parquet(output_dir / "dim_product.parquet", index=False)
dim_store.to_parquet(output_dir / "dim_store.parquet", index=False)

print("Modèle en étoile exporté avec succès")

cle_date : 0 clés distinctes manquantes
id_client : 0 clés distinctes manquantes
id_produit : 1 clés distinctes manquantes
id_magasin : 0 clés distinctes manquantes
Modèle en étoile exporté avec succès


In [21]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

def check_referential_integrity(fact_table, dimensions):

    erreurs = []

    for cle, dimension in dimensions.items():
        orphelines = ~fact_table[cle].isin(dimension[cle])
        nb_orphelines = orphelines.sum()

        print(f"{cle} : {nb_orphelines} clés orphelines")

        if nb_orphelines > 0:
            erreurs.append(f"{cle} : {nb_orphelines}")

    if erreurs:
        raise ValueError(
            "Intégrité référentielle non respectée : "
            + ", ".join(erreurs)
        )

    print("Intégrité référentielle validée")


check_referential_integrity(fact_sales, dimensions)

cle_date : 0 clés orphelines
id_client : 0 clés orphelines
id_produit : 0 clés orphelines
id_magasin : 0 clés orphelines
Intégrité référentielle validée


---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.